In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower, upper, initcap, to_date

spark = SparkSession.builder.appName("DataCleaning").getOrCreate()

# Sample data 
data = [
(101,"Alice","USA","15-01-2023","250","A"),
(102,"Bob","india","01-05-2023","150","B"),
(103,"Charlie","UK","20-02-2023","blank","C"),
(104,"Alice","USA","15-01-2023","250","A"),
(105,"Eve","UK","01-03-2023","300","Blank"),
(106,"Mallory","New York","03-15-2023","400","B"),
(107,"Trent","india","10-04-2023","350","B"),
(108,"Bob","India","05-01-2023","150","B"),
(109,"Oscar","USA","28-02-2023","500","A"),
(110,"Peggy","UK","blank","450","C")
]

columns = ["CustomerID","Name","Country","JoinDate","Sales","Category"]

df = spark.createDataFrame(data, columns)




---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-8670311517140934>, line 67
     60 df = df.withColumn("Category",
     61                    when(col("Category").isNull(), "Unknown")
     62                    .otherwise(upper(col("Category"))))
     64 # ----------------------------
     65 # 6. Final cleaned data
     66 # ----------------------------
---> 67 df.show()
     68 df.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
 

In [0]:
# ----------------------------
# 1. Handle missing values
# ----------------------------
df = df.replace("blank", None).replace("Blank", None)

In [0]:
# ----------------------------
# 2. Standardize text fields
# ----------------------------
df = df.withColumn("Country", initcap(col("Country")))

# Fix incorrect country
df = df.withColumn("Country",
                   when(col("Country") == "New York", "USA")
                   .otherwise(col("Country")))

In [0]:
# 3. Convert data types
# ----------------------------
df = df.withColumn("Sales", col("Sales").cast("int"))

# Handle date formats (two formats)
df = df.withColumn("JoinDate",
    when(col("JoinDate").rlike("^[0-9]{2}-[0-9]{2}-[0-9]{4}$"),
         to_date(col("JoinDate"), "dd-MM-yyyy"))
    .when(col("JoinDate").rlike("^[0-9]{2}-[0-9]{2}-[0-9]{4}$") == False,
         to_date(col("JoinDate"), "MM-dd-yyyy"))
)

In [0]:
# ----------------------------
# 4. Remove duplicates
# ----------------------------
df = df.dropDuplicates()



In [0]:
# 5. Clean Category column
# ----------------------------
df = df.withColumn("Category",
                   when(col("Category").isNull(), "Unknown")
                   .otherwise(upper(col("Category"))))

# ----------------------------
# 6. Final cleaned data
# ----------------------------
df.show()
df.printSchema()

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-8670311517140943>, line 10
      3 df = df.withColumn("Category",
      4                    when(col("Category").isNull(), "Unknown")
      5                    .otherwise(upper(col("Category"))))
      7 # ----------------------------
      8 # 6. Final cleaned data
      9 # ----------------------------
---> 10 df.show()
     11 df.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
 